In [5]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import pos_tag, ne_chunk
from collections import Counter

class TextAnalyzer:
    #A class to perform NLP analysis including tokenization, stemming, 
    #lemmatization, NER, and N-gram analysis.
    
    def __init__(self, file_path):
        #Initializes the analyzer by reading the text from a file.
        #:param file_path: Path to the .txt file.
        self.file_path = file_path
        self.raw_text = self.load_text()
        self.tokens = []
        self.stop_words = set(stopwords.words('english'))

    def load_text(self):
        #Reads content from the text file.
        try:
            with open(self.file_path, 'r', encoding='utf-8') as f:
                return f.read()
        except FileNotFoundError:
            return ""

    def process_tokens(self):
        #Performs tokenization, removes punctuation/stopwords, 
        #and applies stemming and lemmatization.
        #Tokenize and lowercase
        words = word_tokenize(self.raw_text.lower())
        #Filter out non-alphabetic tokens and stopwords
        self.tokens = [w for w in words if w.isalpha() and w not in self.stop_words]
        
        stemmer = PorterStemmer()
        lemmatizer = WordNetLemmatizer()
        
        stems = [stemmer.stem(t) for t in self.tokens]
        lemmas = [lemmatizer.lemmatize(t) for t in self.tokens]
        
        return lemmas #Returning lemmas for frequency analysis

    def get_top_tokens(self, n=20):
        #Returns the n most common lemmas.
        lemmas = self.process_tokens()
        return Counter(lemmas).most_common(n)

    def count_named_entities(self):
        #Identify named entities using NLTK's ne_chunk.
        #Returns the total count of named entities found.
        tags = pos_tag(word_tokenize(self.raw_text))
        chunks = ne_chunk(tags)
        entity_count = 0
        for chunk in chunks:
            if hasattr(chunk, 'label'):
                entity_count += 1
        return entity_count

    def get_ngrams(self, n=3):
        #Generates n-grams from the processed tokens.
        #:param n: The size of the n-gram (default 3 for trigrams).
        #We re-tokenize without lowercasing for better n-gram context
        words = [w.lower() for w in word_tokenize(self.raw_text) if w.isalpha()]
        grams = list(nltk.ngrams(words, n))
        return Counter(grams).most_common(5)

#Execution Logic
files = ["RJ_Lovecraft.txt", "RJ_Tolkein.txt", "RJ_Martin.txt", "Martin.txt"]
analyzers = [TextAnalyzer(f) for f in files]

print(" Comparative Analysis ")
for i, analyzer in enumerate(analyzers[:3]):
    text_num = i + 1
    print("\nAnalysis for Text_%s (%s):" % (text_num, analyzer.file_path))
    print("Top 20 Tokens:", analyzer.get_top_tokens())
    print("Named Entity Count:", analyzer.count_named_entities())

print("\n Trigram Analysis for Authorship ")
for analyzer in analyzers:
    print("\nMost common trigrams in %s:" % analyzer.file_path)
    print(analyzer.get_ngrams(3))

 Comparative Analysis 

Analysis for Text_1 (RJ_Lovecraft.txt):
Top 20 Tokens: [('cosmic', 7), ('eldritch', 5), ('amidst', 5), ('juliet', 5), ('romeo', 5), ('ancient', 4), ('fate', 4), ('mortal', 4), ('love', 4), ('verona', 3), ('whisper', 3), ('forbidden', 3), ('tale', 3), ('bore', 3), ('embrace', 3), ('force', 3), ('city', 2), ('echoed', 2), ('pact', 2), ('tragedy', 2)]
Named Entity Count: 15

Analysis for Text_2 (RJ_Tolkein.txt):
Top 20 Tokens: [('juliet', 7), ('verona', 5), ('amidst', 5), ('romeo', 5), ('land', 4), ('love', 4), ('woven', 3), ('fate', 3), ('upon', 3), ('discord', 3), ('tragic', 3), ('eternal', 3), ('despair', 3), ('among', 2), ('ancient', 2), ('tale', 2), ('thread', 2), ('tapestry', 2), ('fair', 2), ('house', 2)]
Named Entity Count: 23

Analysis for Text_3 (RJ_Martin.txt):
Top 20 Tokens: [('house', 5), ('amidst', 5), ('romeo', 5), ('juliet', 4), ('love', 4), ('passion', 3), ('fate', 3), ('city', 2), ('verona', 2), ('capulet', 2), ('montague', 2), ('hall', 2), ('twis

In [7]:
#Analysis Findings

#Subject Identification: Based on the top tokens ("Juliet", "Romeo", "Verona") and Named Entity Recognition ("Capulet", "Montague"), 
#the subject of the first three texts is Shakespeare's Romeo and Juliet, 
#retold in the styles of H.P. Lovecraft, J.R.R. Tolkien, and George R.R. Martin.

#Authorship of Text 4: Trigram analysis of Martin.txt shows a high frequency of house-based political language ("house", "lord", "son") 
#and grim themes. Comparing the trigrams of RJ_Martin.txt and Martin.txt reveals a shared stylistic preference for "House [Name]" 
#and "Star-crossed" themes, suggesting George R.R. Martin wrote the fourth text.